### Update projected starting lineups

In [ ]:
# from MODELS.scrapStarting import NBADailyLineups

# scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
# scraper.getDict()  # Scrape the lineups
# scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVSV2 import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_50780/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Jalen Johnson,Over,23.5,-137,2025-11-23,2025-11-23T21:31:12Z
1,PrizePicks,player_points,Jalen Johnson,Under,23.5,-137,2025-11-23,2025-11-23T21:31:12Z
2,PrizePicks,player_points,Miles Bridges,Over,21.5,-137,2025-11-23,2025-11-23T21:31:12Z
3,PrizePicks,player_points,Miles Bridges,Under,21.5,-137,2025-11-23,2025-11-23T21:31:12Z
4,PrizePicks,player_points,Kon Knueppel,Over,19.5,-137,2025-11-23,2025-11-23T21:31:12Z


### Top EVs for single bets

In [4]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV%', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
# singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 95 unique players...
Error getting prediction for LeBron James: float division by zero


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV%,KELLY_FRACTION,SIGMA FLAG
624,Rui Hachimura,BetRivers,12.5,15.43,Over,120,0,48.47,0.404,High
664,Dillon Brooks,BetRivers,20.5,23.81,Over,114,0,44.91,0.394,High
225,James Harden,DraftKings,24.5,20.45,Under,-109,1,44.20,0.482,Med
11,Dyson Daniels,FanDuel,12.5,9.64,Under,-106,0,41.63,0.441,Low
116,Austin Reaves,FanDuel,23.5,26.89,Over,102,0,37.72,0.370,High


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


# underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
# underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 54 players...
Error getting prediction for LeBron James: float division by zero
Processing 48 players...
Generated 1054 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
721,James Harden,Dillon Brooks,24.5,18.5,-120,-124,20.45,23.81,under,over,0.752,0.770,0.5673,0.203,0.220,0.277,70.18,0.351,1,5.94,7.19,Med,High,"(8.8, 32.1)","(9.7, 37.9)",0.05,0,70.2
120,Dyson Daniels,Rui Hachimura,12.5,11.5,-130,-115,9.64,15.43,under,over,0.729,0.728,0.5203,0.179,0.178,0.228,56.08,0.280,0,4.70,6.46,Low,High,"(0.4, 18.8)","(2.8, 28.1)",0.05,0,56.1
998,Kevin Love,Harrison Barnes,4.5,13.5,-120,-125,7.45,16.76,over,over,0.726,0.699,0.4974,0.176,0.149,0.205,49.23,0.246,0,4.91,6.25,Low,High,"(0.0, 17.1)","(4.5, 29.0)",0.05,0,49.2
851,Deni Avdija,Jake LaRavia,23.5,7.5,-125,-122,26.88,11.15,over,over,0.697,0.719,0.4915,0.145,0.167,0.196,47.46,0.237,0,6.55,6.29,High,High,"(14.1, 39.7)","(0.0, 23.5)",0.05,0,47.5
427,Brandon Ingram,Donovan Clingan,20.5,9.5,-120,-130,23.75,12.47,over,over,0.682,0.693,0.4631,0.127,0.137,0.164,38.94,0.195,0,6.86,5.91,High,Med,"(10.3, 37.2)","(0.9, 24.1)",0.05,0,38.9
783,Shai Gilgeous-Alexander,Keyonte George,31.5,20.5,-115,-130,29.12,23.03,under,over,0.650,0.651,0.4145,0.099,0.101,0.120,24.35,0.122,0,6.21,6.52,High,High,"(17.0, 41.3)","(10.3, 35.8)",0.05,0,24.3
305,Anthony Black,Ziaire Williams,13.5,10.5,-125,-113,15.81,8.37,over,under,0.637,0.638,0.3982,0.094,0.095,0.111,19.45,0.097,0,6.60,6.04,High,High,"(2.9, 28.7)","(0.0, 20.2)",0.05,0,19.5
582,Noah Clowney,Kawhi Leonard,12.5,18.5,-110,-116,14.66,20.38,over,over,0.630,0.637,0.3929,0.099,0.106,0.120,17.87,0.089,0,6.52,5.38,High,Med,"(1.9, 27.4)","(9.8, 30.9)",0.05,0,17.9
17,Jalen Johnson,Evan Mobley,22.5,19.5,-141,-115,24.50,17.57,over,under,0.621,0.627,0.3820,0.061,0.067,0.076,14.59,0.073,0,6.46,5.93,High,Med,"(11.8, 37.2)","(6.0, 29.2)",0.05,0,14.6
526,Jamal Shead,Jerami Grant,6.5,18.5,-120,-118,5.08,16.45,under,under,0.618,0.619,0.3746,0.074,0.075,0.087,12.37,0.062,0,4.75,6.80,Low,High,"(0.0, 14.4)","(3.1, 29.8)",0.05,0,12.4


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 61 players...
Error getting prediction for LeBron James: float division by zero
Processing 57 players...
Generated 1482 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
10,Tyrese Maxey,Kristaps Porziņģis,31.5,19.5,25.45,15.09,under,under,0.849,0.776,0.6458,0.271,0.198,0.325,93.75,0.469,1,5.85,5.82,Med,Med,"(14.0, 36.9)","(3.7, 26.5)",0.05,0,93.7
111,Bam Adebayo,Brandon Miller,20.5,17.5,15.79,14.01,under,under,0.763,0.769,0.5749,0.185,0.191,0.252,72.46,0.362,0,6.57,4.76,High,Low,"(2.9, 28.7)","(4.7, 23.3)",0.05,0,72.5
1132,Tristan da Silva,Austin Reaves,12.5,22.5,15.52,26.89,over,over,0.678,0.730,0.4847,0.100,0.152,0.160,45.41,0.227,0,6.54,7.18,High,High,"(2.7, 28.3)","(12.8, 41.0)",0.05,0,45.4
945,Dyson Daniels,Kevin Love,11.5,5.0,9.64,7.45,under,over,0.654,0.691,0.4430,0.076,0.113,0.118,32.90,0.165,0,4.70,4.91,Low,Low,"(0.4, 18.8)","(0.0, 17.1)",0.05,0,32.9
331,Andre Drummond,Keyonte George,11.5,20.5,9.31,23.03,under,over,0.645,0.651,0.4119,0.067,0.073,0.086,23.56,0.118,0,5.88,6.52,Med,High,"(0.0, 20.8)","(10.3, 35.8)",0.05,0,23.6
1112,Anthony Black,Marcus Smart,13.5,6.5,15.81,8.76,over,over,0.637,0.647,0.4041,0.059,0.069,0.078,21.23,0.106,0,6.60,5.97,High,Med,"(2.9, 28.7)","(0.0, 20.5)",0.05,0,21.2
501,Justin Edwards,Noah Clowney,9.5,12.5,11.59,14.66,over,over,0.636,0.630,0.3924,0.058,0.052,0.066,17.71,0.089,0,6.03,6.52,High,High,"(0.0, 23.4)","(1.9, 27.4)",0.05,0,17.7
464,Davion Mitchell,Jerami Grant,9.5,18.5,11.48,16.45,over,under,0.624,0.619,0.3783,0.046,0.041,0.052,13.50,0.067,0,6.26,6.80,High,High,"(0.0, 23.7)","(3.1, 29.8)",0.05,0,13.5
594,Dru Smith,Darius Garland,5.5,17.5,6.94,15.51,over,under,0.618,0.616,0.3734,0.040,0.038,0.047,12.02,0.060,0,4.78,6.75,Low,High,"(0.0, 16.3)","(2.3, 28.7)",0.05,0,12.0
896,Zaccharie Risacher,Dean Wade,12.5,6.0,14.34,4.67,over,under,0.616,0.607,0.3665,0.038,0.029,0.040,9.96,0.050,0,6.24,4.88,High,Low,"(2.1, 26.6)","(0.0, 14.2)",0.05,0,10.0


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 73 players...
Error getting prediction for LeBron James: float division by zero
Processing 65 players...
Generated 43105 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
4492,Tyrese Maxey,Brandon Miller,Dillon Brooks,31.5,17.5,18.5,25.45,14.01,23.81,0.849,0.769,0.770,under,under,over,0,171.37,0.343,Med,Low,High
3449,Bam Adebayo,James Harden,Rui Hachimura,21.5,24.5,11.5,15.79,20.45,15.43,0.808,0.752,0.728,under,under,over,0,138.87,0.278,High,Med,High
41516,Deni Avdija,Donovan Clingan,Jake LaRavia,23.5,9.5,7.5,26.88,12.47,11.15,0.697,0.693,0.719,over,over,over,0,87.57,0.175,High,Med,High
35283,Brandon Ingram,Shai Gilgeous-Alexander,Keyonte George,20.5,31.5,20.5,23.75,29.12,23.03,0.682,0.650,0.651,over,under,over,0,55.87,0.112,High,High,High
32876,Anthony Black,Ziaire Williams,Kawhi Leonard,13.5,10.5,18.5,15.81,8.37,20.38,0.637,0.638,0.637,over,under,over,0,39.70,0.079,High,High,Med


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 61 players...
Error getting prediction for LeBron James: float division by zero
Processing 57 players...
Generated 28551 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
682,Tyrese Maxey,Kristaps Porziņģis,Austin Reaves,31.5,19.5,22.5,25.45,15.09,26.89,0.849,0.776,0.730,under,under,over,1,159.69,0.319,Med,Med,High
3620,Bam Adebayo,Brandon Miller,Kevin Love,20.5,17.5,5.0,15.79,14.01,7.45,0.763,0.769,0.691,under,under,over,0,118.95,0.238,High,Low,Low
21956,Dyson Daniels,Tristan da Silva,Keyonte George,11.5,12.5,20.5,9.64,15.52,23.03,0.654,0.678,0.651,under,over,over,0,55.87,0.112,Low,High,High
8983,Andre Drummond,Anthony Black,Marcus Smart,11.5,13.5,6.5,9.31,15.81,8.76,0.645,0.637,0.647,under,over,over,0,43.70,0.087,Med,High,Med
13525,Justin Edwards,Noah Clowney,Jerami Grant,9.5,12.5,18.5,11.59,14.66,16.45,0.636,0.630,0.619,over,over,under,0,33.75,0.067,High,High,High


In [ ]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)